# Basic Evaluation of a Local LLM

*Module 1 · last reviewed: 2026-08*

This notebook is the graded **Part B** of the task. You evaluate a locally running LLM by comparing its answers to expected answers on a small Q&A dataset. You will:

1. **Automatically** query the local model for every question (no manual copy‑paste).
2. Record **token usage** per answer.
3. Score every answer with **three lexical metrics** — token‑overlap Precision/Recall/F1, normalized exact‑match, and answer containment — and compare what each one reveals.

## Prerequisites

- Ollama installed and **running** locally on port `11434`, with the model pulled (`ollama pull llama3.2`). See the task **Prerequisites** for how to install, start, check and stop Ollama.
- Python dependencies from `requirements.txt` (installed in the next cell).

> **Reproducibility note.** The notebook queries the model with `temperature=0` to make answers as stable as possible, but LLM output can still vary slightly between runs.

In [ ]:
# Install pinned dependencies (run once).
%pip install -r requirements.txt

# Alternatively, install directly with pinned versions:
# %pip install pandas==3.0.5 scikit-learn==1.9.0 tiktoken==0.14.0 ollama==0.6.2

: 

In [ ]:
# Imports
import re

import pandas as pd
import ollama
import tiktoken
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
# Configuration
MODEL = "llama3.2"  # pull once with: ollama pull llama3.2
                    # Ollama must be running locally (default endpoint: http://localhost:11434)


def ask_ollama(question: str, model: str = MODEL):
    """Send a question to the local Ollama model.

    Returns (answer, prompt_tokens, answer_tokens). prompt_tokens / answer_tokens are the
    exact token counts reported by the model (prompt_eval_count / eval_count).
    """
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": question}],
        options={"temperature": 0},  # keep answers as deterministic as possible
    )
    answer = response.message.content.strip()
    return answer, response.prompt_eval_count, response.eval_count

In [ ]:
# Fail fast with a clear message if the local Ollama server is not reachable.
try:
    ollama.list()
except Exception as exc:
    raise RuntimeError(
        "Cannot reach Ollama at http://localhost:11434. Start it (launch the Ollama app "
        "or run `ollama serve` in a terminal) and pull the model (`ollama pull llama3.2`), "
        f"then re-run this cell.\nOriginal error: {exc}"
    )
print("Ollama is reachable.")

In [ ]:
# Load the Q&A dataset (columns: Question, Expected Answer, Actual Answer)
questions_df = pd.read_csv("Questions.csv")
questions_df["Question"] = questions_df["Question"].str.strip()  # small data-cleaning step
questions_df.head()

## Generate the actual answers automatically

Instead of copying answers by hand, we query the local model once per question and fill the **Actual Answer** column. We also store `Actual_tokens_model` — the exact number of tokens the model generated (`eval_count`), a real resource‑usage metric.

In [ ]:
answers, prompt_tokens, answer_tokens = [], [], []

for question in questions_df["Question"]:
    answer, p_tok, a_tok = ask_ollama(question)
    answers.append(answer)
    prompt_tokens.append(p_tok)
    answer_tokens.append(a_tok)

questions_df["Actual Answer"] = answers
questions_df["Actual_tokens_model"] = answer_tokens  # exact count from the model (eval_count)
questions_df[["Question", "Actual Answer", "Actual_tokens_model"]].head()

## Counting tokens: exact vs. approximate

There are two useful token counts:

- **Exact (model):** Ollama already returned `eval_count` for every answer — the true number of tokens *this* model produced (stored in `Actual_tokens_model`).
- **Approximate (portable):** the [`tiktoken`](https://github.com/openai/tiktoken) library counts tokens with OpenAI encodings. It uses a **different tokenizer** than Llama, so its counts are an *approximation* for a Llama model — but it is handy for comparing expected vs. actual answers with one consistent tokenizer.

Below we use `tiktoken` to add `Expected_answers_tokens` and `Actual_answers_tokens` (same tokenizer for both, so they are comparable). Comparing these with `Actual_tokens_model` is itself a small data‑quality lesson about tokenizer dependence.

In [ ]:
# Approximate, consistent token counts for expected vs. actual answers (same tokenizer for both).
encoding = tiktoken.encoding_for_model("gpt-4")


def count_tokens(text) -> int:
    if not isinstance(text, str):
        return 0
    return len(encoding.encode(text))


questions_df["Expected_answers_tokens"] = questions_df["Expected Answer"].apply(count_tokens)
questions_df["Actual_answers_tokens"] = questions_df["Actual Answer"].apply(count_tokens)
questions_df[["Expected_answers_tokens", "Actual_answers_tokens", "Actual_tokens_model"]].head()

In [ ]:
questions_df.head()

## Evaluate answers with Precision, Recall and F1

Token counts only tell us *where* answers differ in length, not whether they are correct. For a basic correctness signal we compute **token‑overlap** Precision, Recall and F1 between the expected and actual answers, using `sklearn.metrics`.

### Worked example: Precision for a single answer

**Question:** Who invented the telephone?
- **Expected Answer:** Alexander Graham Bell
- **Actual Answer:** Bell

```python
from sklearn.metrics import precision_score

# 1. Tokenize (lowercase + split)
expected_tokens = "Alexander Graham Bell".lower().split()   # ['alexander', 'graham', 'bell']
actual_tokens = "Bell".lower().split()                      # ['bell']

# 2. Combined vocabulary of unique tokens
all_tokens = list(set(expected_tokens) | set(actual_tokens))

# 3. Binary presence vectors over the vocabulary
y_true = [1 if t in expected_tokens else 0 for t in all_tokens]
y_pred = [1 if t in actual_tokens else 0 for t in all_tokens]

# 4. Precision
precision = precision_score(y_true, y_pred, zero_division=0)
print("Precision:", precision)   # 1.0 — every token the model produced is in the expected answer
```

> **Caveat (read this).** Token‑overlap metrics reward matching *words*, not *meaning*: a correct paraphrase scores low and a wrong answer sharing keywords scores high. Treat these numbers as a cheap first signal, not a verdict on correctness — later modules cover semantic and LLM‑based evaluation. See the Module 1 Metrics Reference.

### Your task: implement three metrics and compare them

You will score each answer with three different **lexical** (word-based) metrics, then compare what each rewards and penalizes:

1. **Token-overlap Precision / Recall / F1** — implement `token_prf`, following the worked example above.
2. **Normalized exact-match** — `normalized_match`: `1` if the answers are identical after normalization, else `0`.
3. **Answer containment** — `answer_contains`: `1` if the expected answer appears *inside* the actual answer (after normalization), else `0`. This credits a correct answer even when the model is verbose.

Implement `token_prf` first (cell below), then the two normalized metrics. A later cell applies all three and builds a comparison table.

In [ ]:
# YOUR CODE HERE
# Compute token-overlap Precision, Recall and F1 for a single (expected, actual) pair,
# following the worked example above.
#
# Steps:
#   1. Lowercase + split Expected and Actual answers into token lists.
#   2. Build the combined vocabulary (union of both token sets).
#   3. Create binary presence vectors y_true (expected) and y_pred (actual).
#   4. Use precision_score / recall_score / f1_score(y_true, y_pred, zero_division=0).


def token_prf(expected: str, actual: str):
    """Return (precision, recall, f1) of token overlap between expected and actual answers."""
    # YOUR CODE HERE
    raise NotImplementedError

### Two more metrics to implement

Token-overlap gives *partial* credit. Now add two stricter, order-independent checks common in QA testing. Both start by **normalizing** the text, so implement `normalize` once and reuse it.

- `normalize(text)` — lower-case, remove punctuation, drop leading articles (`a`, `an`, `the`), and collapse repeated spaces.
- `normalized_match(expected, actual)` → `1` if the normalized strings are exactly equal, else `0`.
- `answer_contains(expected, actual)` → `1` if the normalized expected answer is a substring of the normalized actual answer, else `0`.

> **Heads-up:** with a chatty model, `Exact_match` will be **0 for almost every row** — the model rarely replies with *only* the reference phrase; it wraps it in a sentence. That is itself a finding: exact-match is impractical for free-form LLM output, which is exactly why we also compute `Contains` and token overlap.

In [ ]:
def normalize(text: str) -> str:
    """Lower-case, remove punctuation, drop leading articles (a/an/the), and collapse whitespace."""
    # YOUR CODE HERE
    raise NotImplementedError


def normalized_match(expected: str, actual: str) -> int:
    """Return 1 if the normalized answers are exactly equal, else 0."""
    # YOUR CODE HERE
    raise NotImplementedError


def answer_contains(expected: str, actual: str) -> int:
    """Return 1 if the normalized expected answer appears inside the normalized actual answer, else 0."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# Apply all three metrics across the dataset (leave this cell as is).
questions_df[["Precision", "Recall", "F1_Score"]] = questions_df.apply(
    lambda r: pd.Series(token_prf(r["Expected Answer"], r["Actual Answer"])), axis=1
)
questions_df["Exact_match"] = questions_df.apply(
    lambda r: normalized_match(r["Expected Answer"], r["Actual Answer"]), axis=1
)
questions_df["Contains"] = questions_df.apply(
    lambda r: answer_contains(r["Expected Answer"], r["Actual Answer"]), axis=1
)

# Compare the metrics side by side
questions_df[["Question", "F1_Score", "Exact_match", "Contains"]]

After generating answers and computing the metrics, `questions_df` should have these columns:

| Column | Description |
|---|---|
| `Question` | The question text |
| `Expected Answer` | The expected (reference) answer |
| `Actual Answer` | The model's actual answer |
| `Actual_tokens_model` | Exact tokens generated by the model (`eval_count`) |
| `Expected_answers_tokens` / `Actual_answers_tokens` | Approximate token counts (`tiktoken`) |
| `Precision` / `Recall` / `F1_Score` | Token-overlap metrics |
| `Exact_match` | 1/0 — normalized exact match |
| `Contains` | 1/0 — normalized expected answer contained in the actual answer |

In [ ]:
# Save the evaluated dataset and preview the final table
questions_df.to_csv("Questions_evaluated.csv", index=False)
questions_df

## Your interpretation (graded)

Using the numbers in your comparison table, write 4–7 sentences:

- Find a question where the three metrics **disagree** — e.g., the answer is correct but `F1_Score` is low, or `Contains` is 1 while `Exact_match` is 0. Cite the actual values.
- Explain **why** they disagree: what does each metric reward or penalize?
- Which metric is the most trustworthy signal of correctness **for this Q&A task**, and why?
- All three are *lexical* (word-based). Describe one case where **all three** fail, and propose a **better approach** (e.g., semantic similarity or LLM-as-judge) that would fix it.

*(Write your answer here.)*